In [1]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:90% !important; }</style>"))
import numpy as np
import pandas as pd
import h5py
import os
from tqdm import tqdm 
from scipy import stats
from scipy.io import loadmat

/var/folders/fc/24x3k2m92bvbv7ck1v5mt3n40000gn/T/ipykernel_82300/1016450630.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


In [3]:
## load and do some preprocessing on the CNMFE traces 
def normalize(trace, percentile=True):
    """ Normalize a fluorescence trace by its max or its 99th percentile. """
    trace = trace - np.min(trace)
    if np.percentile(trace, 99) > 0:
        if percentile:
            trace = trace / np.percentile(trace, 99)
        else:
            trace = trace / np.max(trace)
    return trace

def load_and_filter_traces(extract_mat_path: str,
                           labels_mat_path: str,
                           label_key: str = 'labels_ex'
                          ) -> pd.DataFrame:
    """
    Load temporal_weights from extract_mat_path (v7.3) and one of the
    labels_* arrays from labels_mat_path, keep only columns where
    labels == 1, and return as a pandas DataFrame.

    Parameters
    ----------
    extract_mat_path : str
        Path to the v7.3 .mat file containing an 'output/temporal_weights' dataset.
    labels_mat_path : str
        Path to the .mat file (v7.3 or earlier) containing a 'labels' group/struct.
    label_key : str, optional
        Which labels field to use: one of 'labels_ex', 'labels_ml', or
        'labels_overall'. Default is 'labels_ex'.

    Returns
    -------
    pd.DataFrame
        Rows = frames, columns = kept cells (named 'cell_<original_index>').
    """
    # validate choice
    allowed = ('labels_ex','labels_ml','labels_overall')
    if label_key not in allowed:
        raise ValueError(f"label_key must be one of {allowed}, got '{label_key}'")

    # 1) load & transpose temporal_weights → shape (nFrames, nCells)
    with h5py.File(extract_mat_path, 'r') as f:
        tw = f['output']['temporal_weights'][()]  # often (nCells, nFrames)
    tw = np.asarray(tw).T

    # 2) try to load chosen labels via HDF5; if that fails, fall back to loadmat
    try:
        with h5py.File(labels_mat_path, 'r') as f:
            hl = f['labels'][label_key][()]
    except OSError:
        mat = loadmat(labels_mat_path,
                      struct_as_record=False,
                      squeeze_me=True)
        lbl = mat['labels']  # either a dict or a mat_struct
        # pull out the right attribute/key
        if isinstance(lbl, dict):
            hl = lbl[label_key]
        else:
            hl = getattr(lbl, label_key)

    # 3) shape‐check & squeeze
    hl = np.asarray(hl).squeeze()
    if tw.shape[1] != hl.size:
        raise ValueError(
            f"dimension mismatch: temporal_weights is {tw.shape}, "
            f"labels array '{label_key}' has length {hl.size}"
        )

    # 4) filter & build DataFrame
    mask    = (hl == 1)
    kept_i  = np.nonzero(mask)[0]
    filtered = tw[:, mask]
    cols    = [f'cell_{i}' for i in kept_i]

    return (pd.DataFrame(filtered, columns=cols), hl)

def print_h5_tree(name, obj):
    """
    Callback for h5py.File.visititems.
    Prints group/dataset name and, for datasets, its shape and dtype.
    """
    if isinstance(obj, h5py.Group):
        print(f"Group:   {name}/")
    elif isinstance(obj, h5py.Dataset):
        print(f"Dataset: {name}  — shape={obj.shape}, dtype={obj.dtype}")

    
def zScoreTraces(dirPath, CNMFE_real_cells, miniscopeFramesPerSecond):
    
    CNMFE_real_cells = CNMFE_real_cells.apply(pd.to_numeric, errors='coerce')
    C_normalized = CNMFE_real_cells.apply(lambda col: normalize(col), axis=0)
    C_z_scored = CNMFE_real_cells.apply(stats.zscore).set_index(pd.to_timedelta(np.linspace(0, (len(CNMFE_real_cells)-1)*(1/miniscopeFramesPerSecond), len(CNMFE_real_cells)), unit='s'), drop=True)
    C_normalized_z_scored = C_normalized.apply(stats.zscore).set_index(pd.to_timedelta(np.linspace(0, (len(C_normalized)-1)*(1/miniscopeFramesPerSecond), len(C_normalized)), unit='s'), drop=True)

    ##load spatial components by session
    # for v4 dimensions are 600x600 pixels
 
    C_normalized_z_scored.to_csv(dirPath+'_C_traces_filtered_origHz.csv')
    
    print('finished, saved:')
    print(dirPath+'_C_traces_filtered_origHz.csv')
    
    return(C_normalized_z_scored)

def zScoreTraces_withGappedTime(dirPath, CNMFE_real_cells, miniscopeFramesPerSecond, idxsList):
    # Convert all columns to numeric (as before)
    CNMFE_real_cells = CNMFE_real_cells.apply(pd.to_numeric, errors='coerce')
    # 1) Build a list of *all* original frame‐numbers (0…N−1),
    #    then remove the “excised” ranges to get `kept_frames`.
    #    N = M + total_removed
    removed_indices = []
    for start, end in idxsList:
        # remove [start, end) from original
        removed_indices.extend(range(start, end))
    removed_set = set(removed_indices)
    # total frames originally = len(kept) + len(removed)
    total_original_frames = len(CNMFE_real_cells) + len(removed_set)
    # generate all original indices 0…N−1
    all_indices = np.arange(total_original_frames)
    # keep only those not in removed_set, in sorted order
    kept_frames = np.array([i for i in all_indices if i not in removed_set])
    # sanity‐check: kept_frames.size == len(CNMFE_real_cells)
    if kept_frames.shape[0] != len(CNMFE_real_cells):
        raise ValueError(
            "Number of kept frames (%d) != number of rows in CNMFE_real_cells (%d)"
            % (kept_frames.shape[0], len(CNMFE_real_cells))
        )
    # 2) Create a "gapped" timedelta index from those kept_frames:
    #    time (in seconds) = frame_number * (1 / samplingRate)
    times_sec = kept_frames * (1.0 / miniscopeFramesPerSecond)
    time_index = pd.to_timedelta(times_sec, unit='s')
    # 3) Now normalize & z‐score (unchanged from before), but *do not* set the index yet:
    C_normalized = CNMFE_real_cells.apply(lambda col: (col - col.min()) / (col.max() - col.min()), axis=0)
    C_z_scored   = CNMFE_real_cells.apply(stats.zscore, axis=0)
    C_norm_z     = C_normalized.apply(stats.zscore, axis=0)
    # 4) Finally, assign our custom `time_index` to both:
    C_z_scored.index   = time_index
    C_norm_z.index     = time_index
    # 5) (Optional) save to CSV
    C_norm_z.to_csv(dirPath + '_C_traces_filtered_origHz.csv')
    print("Finished, saved to:", dirPath + '_C_traces_filtered_origHz.csv')
    return C_norm_z, C_z_scored

In [37]:
#calcium analysis data - extract and ActSort labels 
dirPath = r'/Users/johnmarshall/Documents/Analysis/miniscope_analysis/miniscopeLinearTrack/Part_2_Day0rec_DMSO_8/'
GCaMPData_EXTRACT_fName = r'1_52_motion_corrected.mat'
ActSortLabels_fName = r'Part2_Day0rec_8_precomputed_1_52_motion_corrected.mat'

#'labels_ex', 'labels_ml', or 'labels_overall'. Default is 'labels_ex'
GCAMP_traces_humanSorted, labels_human = load_and_filter_traces(dirPath+GCaMPData_EXTRACT_fName, dirPath+ActSortLabels_fName, 'labels_ex')
GCAMP_traces_modelSorted, labels_model = load_and_filter_traces(dirPath+GCaMPData_EXTRACT_fName, dirPath+ActSortLabels_fName, 'labels_ml')
GCAMP_traces_OvrallSorted, labels_overall = load_and_filter_traces(dirPath+GCaMPData_EXTRACT_fName, dirPath+ActSortLabels_fName, 'labels_overall')

# if data was "removed" from the calcium signals for the EXTRACT video add that information here, if no data was removed leave as empty list
# videosRemoved = [] 
videosRemoved = [27, 37, 49]; 
videoLength = 1000; 
# video 1 would 0:1000, 
idxsList=[((vidStart*1000)-1000,(vidStart)*1000) for vidStart in videosRemoved]

miniscopeFramesPerSecond = 30

GCAMP_traces_Zscore = zScoreTraces(dirPath, GCAMP_traces_OvrallSorted, miniscopeFramesPerSecond)
GCAMP_traces_ZscoreNormalized_Gapped, GCAMP_traces_Zscore_Gapped  = zScoreTraces_withGappedTime(dirPath, GCAMP_traces_OvrallSorted, miniscopeFramesPerSecond, idxsList)

finished, saved:
/Users/johnmarshall/Documents/Analysis/miniscope_analysis/miniscopeLinearTrack/Part_2_Day0rec_DMSO_8/_C_traces_filtered_origHz.csv
Finished, saved to: /Users/johnmarshall/Documents/Analysis/miniscope_analysis/miniscopeLinearTrack/Part_2_Day0rec_DMSO_8/_C_traces_filtered_origHz.csv


In [38]:
GCAMP_traces_ZscoreNormalized_Gapped

,cell_4,cell_5,cell_6,cell_7,cell_8,cell_9,cell_10,cell_11,cell_12,cell_13,...,cell_232,cell_233,cell_236,cell_239,cell_241,cell_242,cell_246,cell_248,cell_252,cell_254
0 days 00:00:00,0.603335,1.259600,-0.097211,1.871352,-2.132712,0.482260,1.063768,-0.808617,-0.128437,-0.392029,...,-1.880137,0.230291,-0.141987,-0.974901,-0.126094,1.174243,-0.209747,0.529885,-0.479315,-0.092386
0 days 00:00:00.033333333,-0.200197,1.182264,-0.895478,2.403171,-0.386363,-0.693278,1.146219,-1.855244,-0.332357,-1.175271,...,-0.559547,-0.497114,-0.020484,-0.804504,-0.579156,-0.051898,-0.286075,-0.194103,-0.196738,-0.410465
0 days 00:00:00.066666667,-0.051286,0.812882,-1.023856,2.425163,-3.040965,0.430621,-0.067773,-0.954008,-0.875180,-0.960456,...,-2.041188,-0.869181,-1.077083,-0.635941,-0.747729,0.967131,-0.838171,-0.046777,-1.455789,-0.343659
0 days 00:00:00.100000,-0.405695,0.102308,-1.200744,2.143396,-0.750144,-0.581672,0.215996,-0.856238,-0.259201,-1.449508,...,-1.465003,-0.953979,-0.430782,-0.309919,-1.349073,-0.020262,-0.377904,0.481804,-1.497426,-0.421474
0 days 00:00:00.133333333,-0.035390,1.046979,-0.100911,2.125543,-1.232512,0.182043,0.911896,-1.464836,-0.104093,-1.674537,...,-1.866665,-0.084662,-1.124524,-2.071370,-0.500956,0.409416,-0.722999,0.946925,-0.598524,-0.396570
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0 days 00:30:14.600000,-0.621598,-0.116312,-0.200332,0.407947,0.371799,-1.331545,0.695228,0.833916,0.411015,-0.497399,...,0.009395,-0.000080,0.048140,-0.239290,0.174541,1.164781,0.185592,-0.033576,0.074162,0.344499
0 days 00:30:14.633333333,-1.670413,-0.707154,0.651842,0.252504,0.541117,-0.977611,0.370288,0.830703,0.127421,0.179236,...,0.792866,0.241791,-0.074240,-0.556482,1.059433,0.536796,-0.302494,0.180165,0.780446,0.173966
0 days 00:30:14.666666667,0.005881,-0.208353,-0.648365,0.135621,0.419529,0.259242,-0.296247,1.175150,-0.494488,-0.349211,...,-0.094798,0.588737,-0.307436,-0.066797,1.228738,-0.028192,-0.341564,0.529913,-0.458418,-1.045045
0 days 00:30:14.700000,-0.025435,-0.217110,-0.886768,0.897773,-0.705122,0.433847,0.651475,1.061467,-0.972245,-0.104584,...,1.061852,0.228612,0.578846,-0.255242,1.029009,0.405151,0.026635,-0.011177,-0.118505,-0.279486


In [6]:
#behavior analysis info
savePath = r'/Users/johnmarshall/Documents/Analysis/miniscope_analysis/miniscopeLinearTrack/Part_2_Day0rec_DMSO_8/'
sessionPath = r'/Users/johnmarshall/Documents/Analysis/miniscope_analysis/miniscopeLinearTrack/Part_2_Day0rec_DMSO_8/Mouse8Day0_custom_cropped_output_LocationOutput.csv'
behavCamFrameRate = 15
miniscopeCamFrameRate = 30 


In [7]:
#'\\'.join(sessionPath.split(os.sep)[:-1])+'\\timestamp.dat'
print(sessionPath.split(os.sep)[:-1])
# load eZ track output and behavior camera timestamps from miniscope software 
ezTrackOutput = pd.read_csv(sessionPath)
timestampfile = pd.read_table('/'.join(sessionPath.split(os.sep)[:-1])+'/timeStamps.csv', delimiter=',')
miniscope_timestampfile = pd.read_table('/'.join(sessionPath.split(os.sep)[:-1])+'/timeStampsMiniscope.csv', delimiter=',')
    
timestampfile_td = timestampfile.set_index(pd.to_timedelta(np.linspace(0, (len(timestampfile)-1)*(1/behavCamFrameRate), len(timestampfile)), unit='s'), drop=False)
  
miniscopetimestamp_td = miniscope_timestampfile.set_index(pd.to_timedelta(np.linspace(0, (len(miniscope_timestampfile)-1)*(1/miniscopeCamFrameRate), len(miniscope_timestampfile)), unit='s'), drop=False)
    

['', 'Users', 'johnmarshall', 'Documents', 'Analysis', 'miniscope_analysis', 'miniscopeLinearTrack', 'Part_2_Day0rec_DMSO_8']


In [8]:
ezTrackOutput

,File,Location_Thresh,Use_Window,Window_Weight,Window_Size,Start_Frame,Frame,X,Y,Distance_px
0,Mouse8Day0_custom_cropped_output.avi,99.5,True,0.98,60.0,0.0,0,307.998081,7.205374,0.000000
1,Mouse8Day0_custom_cropped_output.avi,99.5,True,0.98,60.0,0.0,1,309.379339,5.951066,1.865788
2,Mouse8Day0_custom_cropped_output.avi,99.5,True,0.98,60.0,0.0,2,309.394783,5.949565,0.015516
3,Mouse8Day0_custom_cropped_output.avi,99.5,True,0.98,60.0,0.0,3,320.386390,10.160735,11.770701
4,Mouse8Day0_custom_cropped_output.avi,99.5,True,0.98,60.0,0.0,4,321.603725,11.908799,2.130172
...,...,...,...,...,...,...,...,...,...,...
26312,Mouse8Day0_custom_cropped_output.avi,99.5,True,0.98,60.0,0.0,26312,43.313695,6.581123,1.226836
26313,Mouse8Day0_custom_cropped_output.avi,99.5,True,0.98,60.0,0.0,26313,41.718205,6.003873,1.696704
26314,Mouse8Day0_custom_cropped_output.avi,99.5,True,0.98,60.0,0.0,26314,41.181252,5.661788,0.636664
26315,Mouse8Day0_custom_cropped_output.avi,99.5,True,0.98,60.0,0.0,26315,38.099449,5.949415,3.095196


In [9]:
timestampfile_td.head()

,Frame Number,Time Stamp (ms),Buffer Index
0 days 00:00:00,0,-20,0
0 days 00:00:00.066666667,1,44,0
0 days 00:00:00.133333333,2,122,0
0 days 00:00:00.200000,3,187,0
0 days 00:00:00.266666667,4,251,0


In [10]:
miniscopetimestamp_td.head()

,Frame Number,Time Stamp (ms),Buffer Index
0 days 00:00:00,0,-26,0
0 days 00:00:00.033333333,1,12,0
0 days 00:00:00.066666667,2,43,0
0 days 00:00:00.100000,3,74,0
0 days 00:00:00.133333333,4,110,0


In [11]:
behavCam_frames = []
sys_clock_behavCam = []
#create "key" for aligning miniscope frames to timestamp file
#then create behavior TD and align
for msCam_frame in tqdm(range(0, len(miniscopetimestamp_td['Frame Number']))):

    #find behav cam frame closest to sys clock time of ms frame
    behavCam_frame = list(timestampfile_td.iloc[(timestampfile_td['Time Stamp (ms)']-miniscopetimestamp_td['Time Stamp (ms)'].iloc[msCam_frame]).abs().argsort()[:1]].index)[0]
    #this is the behavCamIndex that is closest to the corresponding miniscope frame 
    behavCam_frames.append(behavCam_frame)
    #sys clock time of this behavior camera frame 
    sys_clock_behavCam.append(timestampfile_td.loc[behavCam_frame]['Time Stamp (ms)'])

behavCamIdxToAlign = [timestampfile_td.index.get_loc(idx) for idx in behavCam_frames]
    #ezTrackOutput

100%|███████████████████████████████████| 54443/54443 [00:30<00:00, 1798.02it/s]


In [12]:
len(behavCamIdxToAlign)

54443

In [13]:
len(sys_clock_behavCam)

54443

In [14]:
miniscopetimestamp_td['closestBehavCamFrameIdx'] = behavCamIdxToAlign

In [15]:
miniscopetimestamp_td['sys_clock_BehavCamFrame'] = sys_clock_behavCam

In [16]:
miniscopetimestamp_td

,Frame Number,Time Stamp (ms),Buffer Index,closestBehavCamFrameIdx,sys_clock_BehavCamFrame
0 days 00:00:00,0,-26,0,0,-20
0 days 00:00:00.033333333,1,12,0,0,-20
0 days 00:00:00.066666667,2,43,0,1,44
0 days 00:00:00.100000,3,74,0,1,44
0 days 00:00:00.133333333,4,110,0,2,122
...,...,...,...,...,...
0 days 00:30:14.600000,54438,1835609,0,26315,1835627
0 days 00:30:14.633333333,54439,1835642,0,26315,1835627
0 days 00:30:14.666666667,54440,1835671,0,26316,1835692
0 days 00:30:14.700000,54441,1835706,0,26316,1835692


In [17]:
X_coor=[]
Y_coor=[]
Distance_px=[] 

for i in miniscopetimestamp_td['closestBehavCamFrameIdx'].values:
    X_coor.append(ezTrackOutput.loc[i]['X'])
    Y_coor.append(ezTrackOutput.loc[i]['Y'])
    Distance_px.append(ezTrackOutput.loc[i]['Distance_px'])
    
miniscopetimestamp_td['X_coor'] = X_coor
miniscopetimestamp_td['Y_coor'] = Y_coor
miniscopetimestamp_td['Distance_px'] = Distance_px

behavCamDataAligned = miniscopetimestamp_td


In [18]:
miniscopetimestamp_td

,Frame Number,Time Stamp (ms),Buffer Index,closestBehavCamFrameIdx,sys_clock_BehavCamFrame,X_coor,Y_coor,Distance_px
0 days 00:00:00,0,-26,0,0,-20,307.998081,7.205374,0.000000
0 days 00:00:00.033333333,1,12,0,0,-20,307.998081,7.205374,0.000000
0 days 00:00:00.066666667,2,43,0,1,44,309.379339,5.951066,1.865788
0 days 00:00:00.100000,3,74,0,1,44,309.379339,5.951066,1.865788
0 days 00:00:00.133333333,4,110,0,2,122,309.394783,5.949565,0.015516
...,...,...,...,...,...,...,...,...
0 days 00:30:14.600000,54438,1835609,0,26315,1835627,38.099449,5.949415,3.095196
0 days 00:30:14.633333333,54439,1835642,0,26315,1835627,38.099449,5.949415,3.095196
0 days 00:30:14.666666667,54440,1835671,0,26316,1835692,36.932507,6.189738,1.191432
0 days 00:30:14.700000,54441,1835706,0,26316,1835692,36.932507,6.189738,1.191432


In [19]:
# align tracking data to CNMFE for the movies we've analyzed 
# Indicies to "remove" from behav data that were removed from calcium signal data
# Now expand those into a flat list of integer indices:
to_drop = []
for start, end in idxsList:
    to_drop.extend(range(start, end))   # this creates a list of 1000 indices per tuple

# 1) Get the actual index‐labels at those positions:
labels_to_drop = behavCamDataAligned.index[to_drop]
# 2) Drop by label (this preserves the timedelta index for all other rows):
behavCamDataAligned_clean = behavCamDataAligned.drop(labels=labels_to_drop)


In [20]:
idxsList

[(26000, 27000), (36000, 37000), (48000, 49000)]

In [21]:
behavCamDataAligned.iloc[25999:26005]

,Frame Number,Time Stamp (ms),Buffer Index,closestBehavCamFrameIdx,sys_clock_BehavCamFrame,X_coor,Y_coor,Distance_px
0 days 00:14:26.633333333,25999,877288,0,12577,877309,16.562979,11.068273,0.565634
0 days 00:14:26.666666667,26000,877322,0,12577,877309,16.562979,11.068273,0.565634
0 days 00:14:26.700000,26001,877356,0,12578,877370,18.362028,10.581029,1.863862
0 days 00:14:26.733333333,26002,877391,0,12578,877370,18.362028,10.581029,1.863862
0 days 00:14:26.766666667,26003,877422,0,12579,877450,18.380932,10.580058,0.018929
0 days 00:14:26.800000,26004,877456,0,12579,877450,18.380932,10.580058,0.018929


In [22]:
behavCamDataAligned_clean.iloc[25999:26005]

,Frame Number,Time Stamp (ms),Buffer Index,closestBehavCamFrameIdx,sys_clock_BehavCamFrame,X_coor,Y_coor,Distance_px
0 days 00:14:26.633333333,25999,877288,0,12577,877309,16.562979,11.068273,0.565634
0 days 00:15:00,27000,910929,0,13059,910923,18.055068,12.438148,0.419617
0 days 00:15:00.033333333,27001,910963,0,13060,911002,17.509115,11.507458,1.079004
0 days 00:15:00.066666667,27002,910997,0,13060,911002,17.509115,11.507458,1.079004
0 days 00:15:00.100000,27003,911031,0,13060,911002,17.509115,11.507458,1.079004
0 days 00:15:00.133333333,27004,911064,0,13061,911066,16.713722,13.751372,2.380715


In [39]:
GCAMP_traces_ZscoreNormalized_Gapped.iloc[25999:26005]

,cell_4,cell_5,cell_6,cell_7,cell_8,cell_9,cell_10,cell_11,cell_12,cell_13,...,cell_232,cell_233,cell_236,cell_239,cell_241,cell_242,cell_246,cell_248,cell_252,cell_254
0 days 00:14:26.633333333,-1.242204,-0.575521,-0.770077,-0.575419,-0.083573,-0.399238,-0.115894,-0.237232,0.157606,-0.518456,...,-1.116878,-0.509166,-1.052428,-0.235442,-0.101053,-0.138879,0.626436,0.897653,-0.849340,0.215126
0 days 00:15:00,-0.271125,-0.464422,-0.594924,-0.670545,-0.069997,-0.266687,-1.083173,0.745822,0.305308,-0.692228,...,-0.798862,0.051927,-0.310510,-0.254164,0.378125,0.223599,1.212289,0.200503,0.130626,1.190092
0 days 00:15:00.033333333,-0.225099,-0.216051,0.636870,-0.728469,-0.287436,-0.694739,0.345181,0.135652,-0.122741,-0.803016,...,-0.822142,-0.162902,-0.088562,-0.404533,0.153508,0.234516,0.843057,0.216334,0.143101,-0.147927
0 days 00:15:00.066666667,-0.356955,-0.119708,-0.040296,-0.353611,-0.171438,0.168423,0.148437,0.838740,-0.388996,-0.726966,...,0.667122,0.021383,-0.567654,-0.399199,-0.170768,0.904164,0.573469,1.130450,-1.254058,0.039185
0 days 00:15:00.100000,-0.296464,-0.077134,-0.521182,-0.432783,0.396181,0.594368,0.358658,-0.481009,0.597529,-0.710860,...,0.175405,0.842207,-0.274773,0.088483,-0.448833,0.153780,0.390233,1.658262,-1.300089,-0.047056
0 days 00:15:00.133333333,-0.704450,-0.047002,0.053986,-0.999229,-0.783343,0.470451,0.353373,-0.260697,0.282964,-1.047028,...,0.614733,0.882841,-0.003407,-0.588532,-0.210743,0.582747,0.586127,1.448446,0.458095,-0.896791


In [29]:
behavCamDataAligned_clean

,Frame Number,Time Stamp (ms),Buffer Index,closestBehavCamFrameIdx,sys_clock_BehavCamFrame,X_coor,Y_coor,Distance_px
0 days 00:00:00,0,-26,0,0,-20,307.998081,7.205374,0.000000
0 days 00:00:00.033333333,1,12,0,0,-20,307.998081,7.205374,0.000000
0 days 00:00:00.066666667,2,43,0,1,44,309.379339,5.951066,1.865788
0 days 00:00:00.100000,3,74,0,1,44,309.379339,5.951066,1.865788
0 days 00:00:00.133333333,4,110,0,2,122,309.394783,5.949565,0.015516
...,...,...,...,...,...,...,...,...
0 days 00:30:14.600000,54438,1835609,0,26315,1835627,38.099449,5.949415,3.095196
0 days 00:30:14.633333333,54439,1835642,0,26315,1835627,38.099449,5.949415,3.095196
0 days 00:30:14.666666667,54440,1835671,0,26316,1835692,36.932507,6.189738,1.191432
0 days 00:30:14.700000,54441,1835706,0,26316,1835692,36.932507,6.189738,1.191432


In [30]:
GCAMP_traces_Zscore

,cell_4,cell_5,cell_6,cell_7,cell_8,cell_9,cell_10,cell_11,cell_12,cell_13,...,cell_232,cell_233,cell_236,cell_239,cell_241,cell_242,cell_246,cell_248,cell_252,cell_254
0 days 00:00:00,0.603335,1.259600,-0.097211,1.871353,-2.132712,0.482259,1.063767,-0.808616,-0.128438,-0.392029,...,-1.880137,0.230291,-0.141987,-0.974901,-0.126094,1.174243,-0.209746,0.529885,-0.479315,-0.092386
0 days 00:00:00.033333333,-0.200197,1.182264,-0.895479,2.403172,-0.386363,-0.693278,1.146219,-1.855243,-0.332358,-1.175271,...,-0.559547,-0.497114,-0.020484,-0.804504,-0.579156,-0.051897,-0.286075,-0.194104,-0.196738,-0.410466
0 days 00:00:00.066666667,-0.051286,0.812882,-1.023857,2.425164,-3.040965,0.430621,-0.067774,-0.954007,-0.875180,-0.960456,...,-2.041188,-0.869180,-1.077083,-0.635941,-0.747729,0.967131,-0.838170,-0.046778,-1.455789,-0.343659
0 days 00:00:00.100000,-0.405696,0.102308,-1.200745,2.143396,-0.750144,-0.581672,0.215996,-0.856237,-0.259202,-1.449508,...,-1.465002,-0.953978,-0.430782,-0.309919,-1.349073,-0.020261,-0.377904,0.481803,-1.497426,-0.421475
0 days 00:00:00.133333333,-0.035390,1.046979,-0.100911,2.125544,-1.232512,0.182043,0.911895,-1.464836,-0.104093,-1.674537,...,-1.866665,-0.084661,-1.124524,-2.071370,-0.500956,0.409417,-0.722998,0.946924,-0.598524,-0.396570
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0 days 00:28:34.600000,-0.621598,-0.116312,-0.200332,0.407947,0.371799,-1.331545,0.695227,0.833916,0.411015,-0.497399,...,0.009395,-0.000079,0.048140,-0.239290,0.174541,1.164781,0.185592,-0.033576,0.074162,0.344499
0 days 00:28:34.633333333,-1.670413,-0.707154,0.651842,0.252505,0.541118,-0.977611,0.370288,0.830704,0.127421,0.179236,...,0.792866,0.241792,-0.074240,-0.556482,1.059434,0.536796,-0.302493,0.180165,0.780445,0.173965
0 days 00:28:34.666666667,0.005881,-0.208353,-0.648365,0.135622,0.419529,0.259242,-0.296247,1.175151,-0.494489,-0.349211,...,-0.094799,0.588738,-0.307436,-0.066797,1.228739,-0.028192,-0.341563,0.529912,-0.458418,-1.045045
0 days 00:28:34.700000,-0.025436,-0.217110,-0.886769,0.897773,-0.705122,0.433847,0.651475,1.061468,-0.972245,-0.104584,...,1.061852,0.228613,0.578846,-0.255242,1.029009,0.405152,0.026636,-0.011177,-0.118505,-0.279486


In [40]:
CNMFE_aligned = pd.concat([GCAMP_traces_ZscoreNormalized_Gapped, behavCamDataAligned_clean.iloc[0:len(GCAMP_traces_Zscore)]], axis=1)
#CNMFE_aligned.to_csv(dirPath+GCaMPData_EXTRACT_fName.strip(".mat")+'cellTracesAlignedToTracking.csv')
CNMFE_aligned[['Time Stamp (ms)', 'closestBehavCamFrameIdx', 'sys_clock_BehavCamFrame', 'X_coor', 'Y_coor', 'Distance_px']]

,Time Stamp (ms),closestBehavCamFrameIdx,sys_clock_BehavCamFrame,X_coor,Y_coor,Distance_px
0 days 00:00:00,-26,0,-20,307.998081,7.205374,0.000000
0 days 00:00:00.033333333,12,0,-20,307.998081,7.205374,0.000000
0 days 00:00:00.066666667,43,1,44,309.379339,5.951066,1.865788
0 days 00:00:00.100000,74,1,44,309.379339,5.951066,1.865788
0 days 00:00:00.133333333,110,2,122,309.394783,5.949565,0.015516
...,...,...,...,...,...,...
0 days 00:30:14.600000,1835609,26315,1835627,38.099449,5.949415,3.095196
0 days 00:30:14.633333333,1835642,26315,1835627,38.099449,5.949415,3.095196
0 days 00:30:14.666666667,1835671,26316,1835692,36.932507,6.189738,1.191432
0 days 00:30:14.700000,1835706,26316,1835692,36.932507,6.189738,1.191432


In [26]:
behavCamDataAligned_clean

,Frame Number,Time Stamp (ms),Buffer Index,closestBehavCamFrameIdx,sys_clock_BehavCamFrame,X_coor,Y_coor,Distance_px
0 days 00:00:00,0,-26,0,0,-20,307.998081,7.205374,0.000000
0 days 00:00:00.033333333,1,12,0,0,-20,307.998081,7.205374,0.000000
0 days 00:00:00.066666667,2,43,0,1,44,309.379339,5.951066,1.865788
0 days 00:00:00.100000,3,74,0,1,44,309.379339,5.951066,1.865788
0 days 00:00:00.133333333,4,110,0,2,122,309.394783,5.949565,0.015516
...,...,...,...,...,...,...,...,...
0 days 00:30:14.600000,54438,1835609,0,26315,1835627,38.099449,5.949415,3.095196
0 days 00:30:14.633333333,54439,1835642,0,26315,1835627,38.099449,5.949415,3.095196
0 days 00:30:14.666666667,54440,1835671,0,26316,1835692,36.932507,6.189738,1.191432
0 days 00:30:14.700000,54441,1835706,0,26316,1835692,36.932507,6.189738,1.191432


In [ ]:
GCAMP_traces_Zscore

In [ ]:
CNMFE_aligned

In [ ]:
CNMFE_aligned.iloc[25995:26005]

In [ ]:
GCAMP_traces_Zscore

In [ ]:
for videoIDX in range 